In [340]:
import json
import pandas as pd
import os
import itertools

In [341]:
def extract_lines_from_segement(feature):
    temp_lines = pd.DataFrame({
        'line': [line['name'] for line in feature['properties']['lines']]
    }) 
    temp_lines['segmentId'] = feature['properties']['id']
    temp_lines['geometry'] = [feature['geometry']['coordinates']] * len(temp_lines)
    return temp_lines

In [342]:
def extract_segment_geometry(segment):
    for feature in raw_lines['features']:
        if feature['properties']['id'] == segment:
            return feature['geometry']['coordinates']

In [343]:
def concat_segments(segments):
    geom = []
    for segment in segments:
        geom.extend(extract_segment_geometry(segment))
    return geom

In [344]:
def reverse_coords(line_string):
    return [coords[::-1] for coords in line_string]

In [345]:
with open('../data/lines_raw.json', 'r') as in_file:
    raw_lines = json.load(in_file)

In [346]:
lines = pd.concat([extract_lines_from_segement(feature) for feature in raw_lines['features']])
# lines['line'] = lines['line'].apply(lambda x: x.replace('line', '').strip())

In [347]:
lines['start'] = lines['geometry'].apply(lambda geometry: geometry[0])
lines['end'] = lines['geometry'].apply(lambda geometry: geometry[-1])

In [348]:
{line: 'black' for line in lines['line'].unique()}

{'London Overground': 'black',
 'Elizabeth line': 'black',
 'Victoria': 'black',
 'Piccadilly': 'black',
 'District': 'black',
 'Circle': 'black',
 'Metropolitan': 'black',
 'Hammersmith & City': 'black',
 'Central': 'black',
 'Jubilee': 'black',
 'DLR': 'black',
 'Bakerloo': 'black',
 'Northern': 'black',
 'Waterloo & City': 'black',
 'East London': 'black',
 'Thameslink 6tph line': 'black',
 'Tramlink': 'black',
 'Crossrail 2': 'black',
 'IFS Cloud Cable Car': 'black'}

In [349]:
lines = lines[['line', 'segmentId', 'geometry']]
lines['geometry'] = lines['geometry'].apply(str)

In [350]:
# lines = lines.query('line == "District"')

In [351]:
lines = lines.drop_duplicates().reset_index(drop=True)

In [352]:
lines['geometry'] = lines['geometry'].apply(eval)

In [353]:
lines = lines.groupby('line').agg({'geometry': list}).reset_index()

In [354]:
output = {}

In [ ]:
for line in lines['line'].unique():
    line_temp = lines.query(f'line == "{line}"').reset_index(drop=True)
    output[line] = {}
    output[line]['lineName'] = line
    output[line]['geometry'] = {
        'type': 'MultiLineString',
        'coordinates': line_temp['geometry'][0]
    }
    
    # output[line]['segments'] = []
    # for segment in line_temp['segmentId']:
    #     segment_row = line_temp.query(f'segmentId == "{segment}"').reset_index(drop=True)
    #     segment_data = {
    #         'segmentId': segment,
    #         'geometry': {
    #             'type': 'LineString',
    #             'coordinates': eval(segment_row['geometry'][0])
    #         }
    #     }
    #     output[line]['segments'].append(segment_data)

Bakerloo
Central
Circle
Crossrail 2
DLR
District
East London
Elizabeth line
Hammersmith & City
IFS Cloud Cable Car
Jubilee
London Overground
Metropolitan
Northern
Piccadilly
Thameslink 6tph line
Tramlink
Victoria
Waterloo & City


In [356]:
with open('../data/lines.json', 'w') as out_file:
    json.dump(output, out_file)

In [357]:
original_dasharray = [
            [0, 4, 3],
            [0.5, 4, 2.5],
            [1, 4, 2],
            [1.5, 4, 1.5],
            [2, 4, 1],
            [2.5, 4, 0.5],
            [3, 4, 0],
            [0, 0.5, 3, 3.5],
            [0, 1, 3, 3],
            [0, 1.5, 3, 2.5],
            [0, 2, 3, 2],
            [0, 2.5, 3, 1.5],
            [0, 3, 3, 1],
            [0, 3.5, 3, 0.5]
        ]

In [358]:
[
    [num / 2 for num in lst]
    for lst in original_dasharray
]

[[0.0, 2.0, 1.5],
 [0.25, 2.0, 1.25],
 [0.5, 2.0, 1.0],
 [0.75, 2.0, 0.75],
 [1.0, 2.0, 0.5],
 [1.25, 2.0, 0.25],
 [1.5, 2.0, 0.0],
 [0.0, 0.25, 1.5, 1.75],
 [0.0, 0.5, 1.5, 1.5],
 [0.0, 0.75, 1.5, 1.25],
 [0.0, 1.0, 1.5, 1.0],
 [0.0, 1.25, 1.5, 0.75],
 [0.0, 1.5, 1.5, 0.5],
 [0.0, 1.75, 1.5, 0.25]]